In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
from src.db import engine
from src.features import load_results_base, compute_rolling_form

base_df = load_results_base(engine)
print(f"Loaded {len(base_df)} rows")
base_df.head()

Loaded 26759 rows


,result_id,driver_key,forename,surname,constructor_key,constructor_name,race_key,year,round,race_date,grid,position_order,points,status
0,371,1,Lewis,Hamilton,1,McLaren,36,2007,1,2007-03-18,4,3,6.0,Finished
1,392,1,Lewis,Hamilton,1,McLaren,37,2007,2,2007-04-08,4,2,8.0,Finished
2,414,1,Lewis,Hamilton,1,McLaren,38,2007,3,2007-04-15,2,2,8.0,Finished
3,436,1,Lewis,Hamilton,1,McLaren,39,2007,4,2007-05-13,4,2,8.0,Finished
4,458,1,Lewis,Hamilton,1,McLaren,40,2007,5,2007-05-27,2,2,8.0,Finished


In [3]:
featured_df = compute_rolling_form(base_df)

verstappen_2023 = featured_df[
    (featured_df["surname"] == "Verstappen") & (featured_df["year"] == 2023)
][["round", "race_date", "points", "rolling_form_index"]]

verstappen_2023

,round,race_date,points,rolling_form_index
24970,1,2023-03-05,25.0,25.00
24971,2,2023-03-19,19.0,22.00
24972,3,2023-04-02,25.0,23.00
24973,4,2023-04-30,18.0,21.75
24974,5,2023-05-07,26.0,22.60
24975,6,2023-05-28,25.0,22.60
24976,7,2023-06-04,26.0,24.00
24977,8,2023-06-18,25.0,24.00
24978,9,2023-07-02,26.0,25.60
24979,10,2023-07-09,26.0,25.60


In [4]:
from src.features import compute_reliability_rate

reliability_df = compute_reliability_rate(base_df)

redbull_2023 = reliability_df[
    (reliability_df["constructor_name"] == "Red Bull") & (reliability_df["year"] == 2023)
][["forename", "surname", "round", "status", "status_category", "reliability_rate"]]

redbull_2023.tail(10)

,forename,surname,round,status,status_category,reliability_rate
24982,Max,Verstappen,13,Finished,Finished,100.0
24983,Max,Verstappen,14,Finished,Finished,100.0
24984,Max,Verstappen,15,Finished,Finished,100.0
24985,Max,Verstappen,16,Finished,Finished,100.0
24986,Max,Verstappen,17,Finished,Finished,100.0
24987,Max,Verstappen,18,Finished,Finished,100.0
24988,Max,Verstappen,19,Finished,Finished,100.0
24989,Max,Verstappen,20,Finished,Finished,100.0
24990,Max,Verstappen,21,Finished,Finished,100.0
24991,Max,Verstappen,22,Finished,Finished,100.0


In [5]:
redbull_2023.groupby("surname").tail(1)

,forename,surname,round,status,status_category,reliability_rate
23599,Sergio,Pérez,22,Finished,Finished,90.9
24991,Max,Verstappen,22,Finished,Finished,100.0


In [6]:
from src.features import compute_grid_delta

grid_delta_df = compute_grid_delta(base_df)

verstappen_grid = grid_delta_df[
    (grid_delta_df["surname"] == "Verstappen") & (grid_delta_df["year"] == 2023)
][["round", "grid", "position_order", "grid_delta", "grid_delta_running_avg"]]

verstappen_grid

,round,grid,position_order,grid_delta,grid_delta_running_avg
24970,1,1,1,0,0.000000
24971,2,15,2,13,6.500000
24972,3,1,1,0,4.333333
24973,4,2,2,0,3.250000
24974,5,9,1,8,4.200000
24975,6,1,1,0,3.500000
24976,7,1,1,0,3.000000
24977,8,1,1,0,2.625000
24978,9,1,1,0,2.333333
24979,10,1,1,0,2.100000


In [7]:
from src.features import compute_teammate_delta

teammate_df = compute_teammate_delta(base_df)

redbull_teammate_2023 = teammate_df[
    (teammate_df["constructor_name"] == "Red Bull") & (teammate_df["year"] == 2023)
][["round", "surname", "position_order", "position_order_teammate", "teammate_finish_delta"]]

redbull_teammate_2023.head(10)

,round,surname,position_order,position_order_teammate,teammate_finish_delta
61400,1,Pérez,2,1,-1
61402,2,Pérez,1,2,1
61404,3,Pérez,5,1,-4
61406,4,Pérez,1,2,1
61408,5,Pérez,2,1,-1
61410,6,Pérez,16,1,-15
61412,7,Pérez,4,1,-3
61414,8,Pérez,6,1,-5
61416,9,Pérez,3,1,-2
61418,10,Pérez,6,1,-5


In [8]:
feature_store = base_df.copy()

feature_store = compute_rolling_form(feature_store)
feature_store = compute_reliability_rate(feature_store)
feature_store = compute_grid_delta(feature_store)

teammate_features = compute_teammate_delta(base_df)[
    ["driver_key", "race_key", "teammate_finish_delta"]
]

feature_store = feature_store.merge(
    teammate_features, on=["driver_key", "race_key"], how="left"
)

feature_store.to_parquet("../data/processed/feature_store.parquet", index=False)

print(f"Feature store saved: {len(feature_store)} rows, {len(feature_store.columns)} columns")
feature_store.columns.tolist()

Feature store saved: 44335 rows, 23 columns


['result_id',
 'driver_key',
 'forename',
 'surname',
 'constructor_key',
 'constructor_name',
 'race_key',
 'year',
 'round',
 'race_date',
 'grid',
 'position_order',
 'points',
 'status',
 'rolling_form_index',
 'status_category',
 'is_finished',
 'races_so_far',
 'finishes_so_far',
 'reliability_rate',
 'grid_delta',
 'grid_delta_running_avg',
 'teammate_finish_delta']